# Error Analysis
In this Notebook we will explore where our model fails and find out ways to improve our models performance

- Train vs Test Prediction 
- Residual Analysis
- Feature based roc_auc performance
- Feature importance

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time
import random
import statistics
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
sys.path.append('..')  
from src.models.random_guessing import RandomGuessing
from src.models.zero_guessing import ZeroGuessing
from src.evaluation.evaluation_metrics import EvaluationMetrics
from sklearn.metrics import roc_auc_score


from src.data.loader import Loader
from src.data.splitter import split_data

loader = Loader().load()
X_train, X_test, y_train, y_test= split_data(loader.df)

0.49970050648468056
0


In [2]:
model_path = Path.cwd().parent / "src" / "models" / "xgb" / "pipeline.pkl"
pipeline:Pipeline = joblib.load(filename=model_path)

## Train vs Test Set

In this section we compare the models performance on the trainingsset to the test set.
We try to figure out if the model is overfitting the data


In [3]:
y_train_pred = pipeline.predict_proba(X_train)[:, 1]
xgb_guessing_metrics = EvaluationMetrics(y=y_train, y_pred=y_train_pred)

In [4]:
y_test_pred = pipeline.predict_proba(X_test)[:, 1]
xgb_test_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_test_pred)

In [5]:
xgb_guessing_metrics.print_eval_report(headline="Train Set")


Train Set
- Precision: 0.9123319696084161
- Recall: 0.7523344779806013
- F1: 0.8246442368012679
- ROC-AUC: 0.9828018579356741


In [6]:
xgb_test_guessing_metrics.print_eval_report(headline="Test Set")


Test Set
- Precision: 0.7022718214428059
- Recall: 0.43356299212598426
- F1: 0.536132663928191
- ROC-AUC: 0.9250525814977615


The model extremely overfits the training data. 
Train prediction compared to test prediction is singnifficantly better.
To tacke this problem, we will increase the gamma param in the tuning process

## Residual Analysis
in this section we create a new df containing the true value y, our predicted value y^ and its difference

In [7]:
residuals = y_test - y_test_pred

error_df = pd.DataFrame({
    'y': y_test,
    'y_pred': y_test_pred,
    'residual': residuals
})

In [8]:
error_df["residual"].mean()

np.float64(0.00556326903406943)

In [9]:
fraud_residuals   = error_df[error_df.y == 1]["residual"] # residuals for target = 1
legit_residuals   = error_df[error_df.y == 0]["residual"] # residuals for target = 0

print(f"Fraud Mean Residual: {fraud_residuals.mean():.4f}")   # should be extremely positive (constantly undererstimating y)
print(f"Legit Mean Residual: {legit_residuals.mean():.4f}")   


Fraud Mean Residual: 0.6331
Legit Mean Residual: -0.0168


we can see, that the model constantly is underestimating the fraud probability. That lets us conclude that too many fraud payments are nor being detected.


We can see that on average the model is predicting 36% risk for target y's, which is not enough.

## Feature Based ROC analysis
In this section we compare differerent feature valzes and their average ROC score.

This should help us gain insights on what data the modell does good and on what data the modell does bad

### Categorical Features

In [10]:
cat_cols = X_test.select_dtypes(include='object').columns.tolist()

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_11566/3632854767.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_test.select_dtypes(include='object').columns.tolist()


In [11]:
print(cat_cols)

['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [14]:
ohe_cols     = ["ProductCD", "card4", "card6", "DeviceType"]
email_cols   = ["R_emaildomain", "P_emaildomain"]
ordinal_cols = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
                "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", 
                "id_29", "id_30", "id_31", "id_33", "id_34", "id_35", 
                "id_36", "id_37", "id_38", "DeviceInfo"]


In [21]:
results = []

for group_name, cols in [("OHE", ohe_cols), ("Email", email_cols), ("Ordinal", ordinal_cols)]:
    for col in cols:
        for category in X_test[col].dropna().unique():
            mask = (X_test[col] == category).values
            
            if mask.sum() < 1000:
                continue
            if len(np.unique(y_test.values[mask])) < 2:
                continue
                
            auc = roc_auc_score(y_test.values[mask], y_test_pred[mask])
            results.append({
                "encoding":   group_name,
                "feature":    col,
                "category":   category,
                "auc":        auc,
                "n_samples":  mask.sum(),
            })

df_results = pd.DataFrame(results).sort_values("auc")
print(df_results.head(20))  # print worse 20 categorical features


   encoding        feature             category       auc  n_samples
59  Ordinal          id_31  ie 11.0 for desktop  0.838370       1175
55  Ordinal          id_30            Windows 7  0.839094       1601
33  Ordinal             M5                    T  0.871822      22982
7       OHE          card4             discover  0.872006       1257
40  Ordinal             M8                    T  0.873909      25642
36  Ordinal             M6                    F  0.885295      48855
41  Ordinal             M9                    T  0.888912      59924
37  Ordinal             M7                    F  0.889062      62248
42  Ordinal             M9                    F  0.891005      11906
26  Ordinal             M2                    T  0.892027      75461
28  Ordinal             M3                    T  0.892929      66951
2       OHE      ProductCD                    W  0.893214      93669
39  Ordinal             M8                    F  0.895278      46188
64  Ordinal          id_31   mobil

we can clearly see certain features performing worse than others. Most of the features however, have little to no relevance for the model (m1-m9).

We can clearly see that Procudt W is performing worse than other products of the same column. 

In [18]:
df_results.groupby("encoding")["auc"].mean()

encoding
Email      0.909203
OHE        0.919591
Ordinal    0.909522
Name: auc, dtype: float64

we can clearly see small to no difference at all comparing the different feature engineering methods performed on the categorical features